# Tính Recall@5 và NDCG@5 trên dữ liệu clickout mẫu
Notebook tạo 10 clickout giả lập. Mỗi clickout có một khách sạn được click. Hàm `evaluate` bên dưới được lấy nguyên văn từ code train.

In [ ]:
import numpy as np

# Adapter NumPy nhỏ để chạy nguyên hàm train trên máy chưa cài PyTorch.
class DemoTensor:
    def __init__(self, data):
        self.data = np.asarray(data)
        self.device = 'cpu'
    def __getitem__(self, index):
        return DemoTensor(self.data[index])
    def item(self):
        return self.data.item()
    def index_select(self, dim, indices):
        return DemoTensor(np.take(self.data, indices.data, axis=dim))
    def __gt__(self, other):
        return DemoTensor(self.data > getattr(other, 'data', other))
    def eq(self, other):
        return DemoTensor(self.data == getattr(other, 'data', other))
    def lt(self, other):
        return DemoTensor(self.data < getattr(other, 'data', other))
    def __and__(self, other):
        return DemoTensor(self.data & other.data)
    def sum(self):
        return DemoTensor(self.data.sum())

class DemoTorch:
    long = np.int64
    float32 = np.float32
    @staticmethod
    def tensor(data, dtype=None):
        return DemoTensor(np.asarray(data, dtype=dtype))
    @staticmethod
    def as_tensor(data, device=None, dtype=None):
        return DemoTensor(np.asarray(data, dtype=dtype))
    @staticmethod
    def arange(n, device=None):
        return DemoTensor(np.arange(n))

torch = DemoTorch()

# Alias phục vụ type annotations giống file train.
MLEUP = object
Data = list

class DemoModel:
    def eval(self):
        return self

def batch_at(dataset, indices):
    return [dataset[i] for i in indices]

def forward(model, batch):
    targets = torch.tensor([HOTEL_ID[row['target_id']] for row in batch], dtype=torch.long)
    scores = torch.tensor([row['scores'] for row in batch], dtype=torch.float32)
    return targets, scores, None

def check_time(started, max_hours):
    return None  # Không giới hạn thời gian trong bộ dữ liệu demo.


In [ ]:
# Mỗi record là một clickout; mỗi clickout có đúng một khách sạn được click.
HOTEL_ID = {name: index for index, name in enumerate('ABCDEF')}
DATASET = [
    {'session_id': 1, 'target_id': 'A', 'scores': [0.92, 0.85, 0.70, 0.64, 0.51, 0.40]},
    {'session_id': 2, 'target_id': 'A', 'scores': [0.92, 0.85, 0.70, 0.64, 0.51, 0.40]},
    {'session_id': 3, 'target_id': 'A', 'scores': [0.92, 0.85, 0.70, 0.64, 0.51, 0.40]},
    {'session_id': 4, 'target_id': 'A', 'scores': [0.92, 0.85, 0.70, 0.64, 0.51, 0.40]},
    {'session_id': 5, 'target_id': 'A', 'scores': [0.92, 0.85, 0.70, 0.64, 0.51, 0.40]},
    {'session_id': 6, 'target_id': 'A', 'scores': [0.90, 0.75, 0.75, 0.64, 0.51, 0.40]},
    {'session_id': 7, 'target_id': 'A', 'scores': [0.90, 0.75, 0.75, 0.64, 0.51, 0.40]},
    {'session_id': 8, 'target_id': 'F', 'scores': [0.92, 0.85, 0.70, 0.64, 0.51, 0.40]},
    {'session_id': 9, 'target_id': 'F', 'scores': [0.92, 0.85, 0.70, 0.64, 0.51, 0.40]},
    {'session_id': 10, 'target_id': 'F', 'scores': [0.92, 0.85, 0.70, 0.64, 0.51, 0.40]},
]
CANDIDATES = [
    ['A', 'B', 'C', 'D', 'E', 'F'], ['A', 'B', 'C', 'D', 'E', 'F'],
    ['A', 'B', 'C', 'D', 'E', 'F'], ['A', 'B', 'C', 'D', 'E', 'F'],
    ['A', 'B', 'C', 'D', 'E', 'F'], ['A', 'B', 'C', 'D', 'E', 'F'],
    ['A', 'B', 'C', 'D', 'E', 'F'], ['C', 'A', 'F', 'B', 'E', 'D'],
    ['F', 'E', 'D', 'C', 'B', 'A'], ['A', 'B', 'C', 'D', 'E', 'F'],
]
print('Clickout:', len(DATASET), '| khách sạn mục tiêu mỗi clickout: 1')
print('Clickout | Khách sạn được click | Danh sách ứng viên')
for row, candidate_ids in zip(DATASET, CANDIDATES):
    candidate_labels = ', '.join(candidate_ids)
    print(f"{row['session_id']:>8} | {row['target_id']:>20} | {candidate_labels}")

def evaluate_labeled_data(model, dataset, candidates, batch_size):
    # Đổi nhãn A-F sang ID chỉ số mà mô hình dùng để tra tensor điểm.
    numeric_candidates = [[HOTEL_ID[name] for name in slate] for slate in candidates]
    return evaluate(model, dataset, numeric_candidates, batch_size)


Dữ liệu mỗi lượt gồm danh sách khách sạn được hiển thị, điểm dự đoán và một khách sạn mục tiêu đã được click. Hàm đánh giá tính rank theo đúng cách xử lý điểm hòa trong code train, sau đó cộng Recall@5 và NDCG@5.

In [ ]:
def evaluate(model: MLEUP, dataset: Data, candidates: list, batch_size: int,
             started: float = 0, max_hours: float = 0) -> dict:
    if len(candidates) != len(dataset):
        raise ValueError("Candidate lists must align with the evaluation examples")
    model.eval()
    count = 0
    recall5 = 0.0
    ndcg5 = 0.0
    for offset in range(0, len(dataset), batch_size):
        indices = list(range(offset, min(offset + batch_size, len(dataset))))
        targets, scores, _ = forward(model, batch_at(dataset, indices))
        for row, source_index in enumerate(indices):
            item_ids = candidates[source_index]
            target_id = int(targets[row].item())
            target_position = item_ids.index(target_id)
            positions = torch.as_tensor(item_ids, device=scores.device, dtype=torch.long)
            values = scores[row].index_select(0, positions)
            target_score = values[target_position]
            before = torch.arange(len(item_ids), device=scores.device).lt(target_position)
            rank = 1 + (values > target_score).sum().item() + (
                values.eq(target_score) & before
            ).sum().item()
            count += 1
            # One relevant target per clickout: Recall@5 equals Hit@5.
            recall5 += int(rank <= 5)
            ndcg5 += 1.0 / np.log2(rank + 1) if rank <= 5 else 0.0
        if offset % (batch_size * 500) == 0:
            print(f"  eval: {offset:,}/{len(dataset):,}", flush=True)
        if started:
            check_time(started, max_hours)
    if not count:
        raise ValueError("Cannot evaluate an empty split")
    return {"ndcg_at_5": ndcg5 / count,
            "recall_at_5": recall5 / count,
            "examples": count}

In [ ]:
# Đưa dữ liệu clickout A-F qua nguyên hàm evaluate từ code train.
metrics = evaluate_labeled_data(DemoModel(), DATASET, CANDIDATES, batch_size=4)
print('Kết quả metric:', metrics)
